# Chapter 3: Transformer Core Components

Self-Attention is just the engine. A Transformer is the whole car. 
Let's look at the actual architecture piece-by-piece, exactly as it appears in modern LLMs.

---

## 1. Tokenization (Turning Words to Chunks)

Before reaching the embedding layer, we don't just split sentences by spaces. Standard string splitting results in massive dictionaries (millions of words), and fails on typos.

We use **Sub-word Tokenization**:
It breaks "unbelievable" into "un" + "believ" + "able".

### The Big Three Algorithms
1. **BPE (Byte Pair Encoding):** Used by GPT/LLaMA. It starts with individual characters, constantly counting which pairs occur most often (e.g., 'e' and 'r' -> 'er'), and merges them until it hits a target vocab size (like 50,000).
2. **WordPiece:** Used by BERT. Very similar to BPE but uses a slightly different statistical likelihood metric. Words not at the start have a `##` prefix (e.g., `##ing`).
3. **SentencePiece:** Used by T5/Google. BPE assumes you have "words" separated by spaces. SentencePiece treats the raw input as just bytes (spaces are a character `_`). This is crucial because **not all languages use spaces!** (e.g., Chinese, Japanese).

---

## 2. Positional Embeddings (Where am I?)

**The Problem:** Our Attention mechanism processes ALL words simultaneously (in parallel matrices). It fundamentally has **no concept of word order.**
- "The dog bit the man" = "The man bit the dog" mathematically. 
We must artificially inject position information.

### Types of Positional Injection
1. **Absolute (Original Transformers):** Simply add a sine/cosine wave fixed position vector to the word embedding vector. E.g., "Apple" gets vector $A$. Pos=1 gets sine vector $P_1$. Final = $A + P_1$.
2. **Relative (T5):** Don't memorize "Position 3". Just memorize "Word A is exactly 2 spaces away from Word B".
3. **RoPE (Rotary Positional Embedding - LLaMA):** Instead of *adding* a vector, what if we *rotate* the vector like a dial in 500-dimensional space depending on its position? Extrapolates to longer sequences very well.
4. **ALiBi (Attention with Linear Biases - Bloom):** Completely ditches vectors. Just mathematically penalize the final Attention Score if two words are far apart.

---

## 3. Multi-Head Attention (MHA)

Why just calculate Attention once?

What if we split the word embedding into 8 smaller chunks (heads)? 
Each head gets its own Query, Key, and Value matrices. They run in parallel.

**Intuition:**
We want the network to look at the sentence from different perspectives simultaneously.
- **Head 1:** Focuses on "Who is doing the action?" (Syntax)
- **Head 2:** Focuses on "What emotion is expressed?" (Sentiment)
- **Head 3:** Focuses on "Does this word rhyme?" (Rhyme)

After all $N$ heads finish, we concatenate them back together and pass them through a linear layer.

---

## 4. Feed-Forward Networks (FFN)

Under every Attention block is a standard, dense, fully connected Multi-Layer Perceptron (usually two linear layers with a non-linearity like GeLU in between).

**Why?**
Attention is just mixing and moving data around. It's fundamentally linear routing.
We need **non-linearity** to actually "think" and process complex logic. The FFN acts as massive key-value memory banks storing factual knowledge.
*(Recent papers suggest the MLP layers act as the "database" while Attention acts as the "search engine" routing the query to the correct MLP).* 

---

## 5. Layer Normalization & Residual Connections

Deep neural networks struggle with vanishing gradients. If we have 96 layers (like GPT-3), the signal dies.

### Residual (Add) Connections
Instead of Output = `F(Input)`
We do Output = `Input + F(Input)`
The original signal bypasses the block entirely. The block only learns the "residual" difference it needs to add. This guarantees the gradient flows cleanly backward without vanishing.

### Layer Normalization: Pre-Norm vs. Post-Norm
We must normalize the values (mean=0, variance=1) so the network stays stable.

**Post-Norm (Original 2017 Paper):**
`Output = LayerNorm( x + Sublayer(x) )`
*(Normalization happens AFTER adding the residual).* 
- **Problem:** The gradients get mangled through 100 LayerNorms going backward. You need massive "Learning Rate Warmup" hacks to stop it from exploding at the start of training.

**Pre-Norm (Modern LLMs like GPT/LLaMA):**
`Output = x + Sublayer( LayerNorm(x) )`
*(Normalization happens inside the sublayer, untouched residual passes by).* 
- **Fix:** The clean residual highway has literally NO normalizations on it! Gradients flow flawlessly from the top layer to the bottom layer. Training is incredibly stable.

---

## 🎓 Interview Q&A

**Q: What is the primary difference between WordPiece and BPE tokenization?**  
A: BPE merges pairs purely based on highest frequency counts. WordPiece merges based on a likelihood score (prob(pair) / (prob(a) * prob(b))). This means WordPiece looks for combinations that frequently appear together but rarely appear separately.

**Q: Why do modern LLMs use Pre-Norm instead of Post-Norm found in the original Attention Is All You Need paper?**  
A: Post-normalization normalizes the entire output of the residual connection, meaning gradients flowing backwards hit every single LayerNorm layer, degrading the signal in Deep Networks and causing massive instability early in training. Pre-norm normalizes the input before the sub-layer but leaves the residual addition untouched, creating a perfect un-normalized gradient highway back to the first layer.

**Q: Explain RoPE (Rotary Position Embeddings) simply.**  
A: Instead of treating position as an additive vector like absolute embeddings, RoPE treats pairs of features as coordinates on a 2D plane and literally rotates them by an angle directly proportional to their sequence position. The math works out such that the dot product of any two words (their attention score) only depends on the *relative distance between them*, making it far better at extrapolating to sequence lengths longer than what it was trained on.